In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from surprise import Reader, Dataset, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

# =========================================================
# 1. 파일 경로 설정 및 데이터 로드
# =========================================================
BASE_DIR = '../'
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
META_PATH = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
REVIEWS_PATH = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')

# 파일 로드
meta_df = pd.read_pickle(META_PATH)
reviews_df = pd.read_csv(REVIEWS_PATH)

# =========================================================
# 2. 악기 도메인(Train) / 음악 도메인(Test) 분리
# =========================================================
asin_instruments = meta_df.loc[meta_df["domain"] == "instruments", "asin"].unique()
asin_music = meta_df.loc[meta_df["domain"] == "music", "asin"].unique()

reviews_instruments = reviews_df[reviews_df["asin"].isin(asin_instruments)]
reviews_music = reviews_df[reviews_df["asin"].isin(asin_music)]


if len(reviews_instruments) == 0:
    raise ValueError("")

# =========================================================
# 3. surprise Dataset 생성
# =========================================================
reader = Reader(rating_scale=(1.0, 5.0))
train_data = Dataset.load_from_df(
    reviews_instruments[['reviewerID', 'asin', 'overall']], reader
)
test_data = Dataset.load_from_df(
    reviews_music[['reviewerID', 'asin', 'overall']], reader
)
trainset = train_data.build_full_trainset()
testset = test_data.build_full_trainset().build_testset()


# =========================================================
# 4. SVD 학습
# =========================================================
svd_model = SVD(n_factors=50, n_epochs=15, random_state=42)
svd_model.fit(trainset)

# =========================================================
# 5. 음악 도메인 데이터로 예측 및 평가
# =========================================================
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae = accuracy.mae(predictions, verbose=False)
print(f"\n🎵 음악 도메인 예측 결과 (악기로 학습한 모델 평가)")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")

# =========================================================
# 6. 예시 예측
# =========================================================
sample_user = testset[0][0]
sample_item = testset[0][1]
pred = svd_model.predict(sample_user, sample_item)
print(f"\n[예시 예측]")
print(f"사용자 ID : {sample_user}")
print(f"테스트 아이템 ASIN : {sample_item}")
print(f"예측 평점 : {pred.est:.4f} / 5.0")

# =========================================================
# 7. 사용자별 Top-10 추천 생성 함수 (표 형태 출력)
# =========================================================
from collections import defaultdict
import pandas as pd

def get_top_n(predictions, n=10):
    """각 사용자별 예측 평점이 가장 높은 Top-N 아이템을 반환"""
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

# 사용자별 Top-3 생성
top_n = get_top_n(predictions, n=3)

# 예시 사용자 1명 선택
example_user = list(top_n.keys())[0]
print(f"\n[🎯 사용자 {example_user}의 Top-3 추천 목록]\n{'-'*70}")


# 보기 좋은 표로 정리
rec_list = []
for rank, (iid, score) in enumerate(top_n[example_user], start=1):
    title = meta_df.loc[meta_df["asin"] == iid, "title_str"].values
    title = title[0] if len(title) > 0 else "(제목 정보 없음)"
    rec_list.append([rank, title, iid, round(score, 3)])

rec_df = pd.DataFrame(rec_list, columns=["순위", "제목", "ASIN", "예측 평점"])
print(rec_df.to_markdown(index=False))
# =========================================================
# 8. 모델 저장
# =========================================================
model_save_path = os.path.join(OUTPUT_DIR, 'svd_collaborative_instrument2music.joblib')
joblib.dump(svd_model, model_save_path)
print(f"\n💾 모델 저장 완료: {model_save_path}")



ParserError: Error tokenizing data. C error: Expected 14 fields in line 6, saw 15
